# AccessFlow — priority model training (Module 2)

Offline experiment that trains and evaluates the **High / Medium / Low** priority classifier and exports the production artifacts.
It is **not** run by the FastAPI service; production only loads the exported `.joblib`.

**Read before running**
- Labels must come from the human annotation process in `data/ANNOTATION_GUIDELINE.md`. axe-core impact is **not** the label.
- Results are only meaningful if the dataset quality gates pass (`data/reports/quality_<version>.json`). Otherwise the exported model goes to `artifacts/dev/` and is flagged as not valid performance.
- With `USE_SYNTHETIC_FIXTURE = True` the notebook runs on made-up rows to test the code. **Its metrics must never be reported.**

Sections: 1 setup · 2 load · 3 inspect · 4 missing/duplicates · 5 label distribution · 6 features · 7 split · 8 leakage · 9 preprocessing · 10 baselines · 11 random forest · 12 tuning · 13 comparison · 14 test metrics · 15 confusion matrix · 16 classification report · 17 error analysis · 18 export

## 1. Imports and environment setup

In [ ]:
import sys, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from sklearn.metrics import classification_report
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from accessflow_ml import data, features, train

SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2. Dataset loading
The notebook never substitutes synthetic data silently: if the real dataset file is missing it stops.

In [ ]:
DATASET_VERSION = "v0.1"
USE_SYNTHETIC_FIXTURE = False        # True ONLY to smoke-test this notebook. Never report those results.

if USE_SYNTHETIC_FIXTURE:
    DATASET_PATH = ROOT / "tests/fixtures/synthetic_fixture.jsonl"
    print("!" * 78, "\n  SYNTHETIC FIXTURE: results below are NOT real model performance.\n" + "!" * 78)
else:
    DATASET_PATH = ROOT / f"data/dataset_{DATASET_VERSION}.jsonl"
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f"{DATASET_PATH} not found. Collect + annotate data and run tools/build_dataset.py first.")
ARTIFACTS_ROOT = ROOT / "artifacts"

records = data.read_jsonl(DATASET_PATH)
df_raw = pd.DataFrame(records)
print(f"loaded {len(df_raw)} findings from {DATASET_PATH.name}; data_origin = {sorted(df_raw['data_origin'].unique())}")

## 3. Dataset inspection

In [ ]:
print("shape:", df_raw.shape)
print("domains:", df_raw["registrable_domain"].nunique(), "| rules:", df_raw["rule_id"].nunique())
display(df_raw[["finding_id", "registrable_domain", "rule_id", "axe_impact", "node_count", "final_label", "split"]].head(8))
display(df_raw["node_count"].describe())

## 4. Missing values and duplicates

In [ ]:
required = ["finding_id", "rule_id", "node_count", "registrable_domain", "final_label", "split", "data_origin"]
print("missing (NaN) per required column:\n", df_raw[required].isna().sum().to_string())
print("\nempty strings per required column:\n", (df_raw[required] == "").sum().to_string())

dup_id = int(df_raw.duplicated("finding_id").sum())
dup_sem = int(df_raw.duplicated(["registrable_domain", "rule_id", "dedupe_key"]).sum())
print(f"\nduplicate finding_id: {dup_id} | duplicate (domain, rule, dedupe_key): {dup_sem}")

clean = df_raw.drop_duplicates("finding_id").drop_duplicates(["registrable_domain", "rule_id", "dedupe_key"])
removed = len(df_raw) - len(clean)
records = [r for r in records if r["finding_id"] in set(clean["finding_id"])]
seen = set(); records = [r for r in records if not (r["finding_id"] in seen or seen.add(r["finding_id"]))]
print(f"removed {removed} duplicate rows; {len(records)} remain")
if removed:
    print("NOTE: duplicates should be fixed upstream (tools/build_dataset.py); the quality gate will still fail.")

## 5. Label distribution and class imbalance

In [ ]:
labels_ct = pd.Series([r["final_label"] for r in records]).value_counts().reindex(features.LABELS, fill_value=0)
print(labels_ct.to_string())
print("imbalance ratio (max/min class):", round(labels_ct.max() / max(labels_ct.min(), 1), 2))
split_ct = pd.crosstab(pd.Series([r["split"] for r in records], name="split"), pd.Series([r["final_label"] for r in records], name="label"))[features.LABELS]
display(split_ct)
fig, ax = plt.subplots(figsize=(5, 3)); labels_ct.plot.bar(ax=ax, color="#4c72b0"); ax.set_title("Label distribution"); ax.set_ylabel("findings"); plt.tight_layout(); plt.show()
print("Imbalance handling: class_weight='balanced' in every ML model; macro-F1 and High recall are the selection/headline metrics; no resampling before the split.")

## 6. Feature engineering
Features come from `accessflow_ml.features.extract_features` — the **same code** the FastAPI service uses. No raw HTML, selector, URL, domain or label is used.

In [ ]:
d = train.load_dataset(records)      # validates labels/splits/origin and builds the feature frame
display(d["X"].head())
display(pd.DataFrame(features.feature_schema()["features"])[["name", "kind", "description"]])

## 7. Train / validation / test strategy
- **Test:** whole *domains* are assigned to test (hash of seed+domain, locked in `data/splits/split_manifest.json`). It is used **once**, after model selection.
- **Validation:** `StratifiedGroupKFold` on the train split, grouped by domain, stratified by label. Hyper-parameters and the final model are chosen here.

In [ ]:
print("rows   train/test:", len(d["y_train"]), "/", len(d["y_test"]))
print("domains train/test:", len(set(d["g_train"])), "/", len(set(d["g_test"])))
assert not set(d["g_train"]) & set(d["g_test"]), "a domain appears in both splits"
cv = train.cv_splits(d["y_train"], d["g_train"], SEED)
print("CV:", type(cv).__name__, "with", cv.n_splits, "folds (grouped by domain)")

## 8. Data leakage prevention

In [ ]:
assert not features.FORBIDDEN_FEATURES & set(features.FEATURE_COLUMNS)
print("forbidden-in-features check passed:", sorted(features.FORBIDDEN_FEATURES))
# Diagnostic (TRAIN only): how much does the human label simply follow axe impact? High agreement means the model may add little.
mapped = pd.Series(d["X_train"]["axe_impact"]).map(features.FALLBACK_IMPACT_TO_PRIORITY)
agreement = float((mapped.to_numpy() == d["y_train"]).mean())
print(f"label == impact-mapping on train: {agreement:.1%}")
display(pd.crosstab(d["X_train"]["axe_impact"], pd.Series(d["y_train"], name="label")))

## 9. Preprocessing and encoding
One-hot for categoricals (unknown values ignored at inference), standard scaling for numerics. It is fitted **inside** each CV fold / on train only, as part of the model pipeline.

In [ ]:
pre = features.build_preprocessor().fit(d["X_train"])
print("encoded feature matrix:", pre.transform(d["X_train"]).shape)
print(list(pre.get_feature_names_out())[:12], "...")

## 10. Baseline classifiers
Majority class, the fixed **axe-impact → priority mapping**, logistic regression and a decision tree. All tuned/evaluated with the same grouped CV on train.

In [ ]:
cands = train.candidates(SEED)
fitted, cv_summary = {}, {}
def fit_one(name):
    pipe, grid = cands[name]
    fitted[name], cv_summary[name] = train.tune_candidate(pipe, grid, d["X_train"], d["y_train"], d["g_train"], SEED)
    c = cv_summary[name]
    print(f"{name:22s} CV macro-F1 {c['cv_macro_f1_mean']:.3f} ± {c['cv_macro_f1_std']:.3f} | High recall {c['cv_high_recall_mean']:.3f} | {c['best_params']}")
for n in ["majority_class", "axe_impact_mapping", "logistic_regression", "decision_tree"]:
    fit_one(n)

## 11–12. Random Forest training and hyper-parameter tuning
Grid search over depth, leaf size and `max_features` with `class_weight='balanced_subsample'`, scored by grouped-CV macro-F1 (High recall is also recorded).

In [ ]:
fit_one("random_forest")
print("grid:", cands["random_forest"][1])

## 13. Model comparison (cross-validation on train only)

In [ ]:
cmp = pd.DataFrame(cv_summary).T[["cv_macro_f1_mean", "cv_macro_f1_std", "cv_high_recall_mean", "cv_folds"]].sort_values("cv_macro_f1_mean", ascending=False)
display(cmp)
best = train.select_best(cv_summary)     # export candidate chosen by CV, never by the test split
print("selected for export:", best, "| Random Forest selected:", best == "random_forest")

## 14. Precision, recall and F1 on the held-out test split
Evaluated **once**, now that the model is fixed.

In [ ]:
test_metrics = {n: train.evaluate_model(m, d["X_test"], d["y_test"]) for n, m in fitted.items()}
tbl = pd.DataFrame({n: {"precision_macro": m["precision_macro"], "recall_macro": m["recall_macro"], "f1_macro": m["f1_macro"], "High_recall": m["high_recall"]} for n, m in test_metrics.items()}).T
display(tbl.sort_values("f1_macro", ascending=False))
ci = train.cluster_bootstrap_ci(fitted[best], d["X_test"], d["y_test"], d["g_test"], n_boot=1000, seed=SEED)
print("selected model 95% CI (domain bootstrap):", ci)
print("beats axe-impact mapping baseline (test macro-F1):", test_metrics[best]["f1_macro"] > test_metrics["axe_impact_mapping"]["f1_macro"])

## 15. Confusion matrix (selected model)

In [ ]:
cm = np.array(test_metrics[best]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4.2, 3.6)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), features.LABELS); ax.set_yticks(range(3), features.LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix — {best}")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

## 16. Classification report

In [ ]:
print(classification_report(d["y_test"], fitted[best].predict(d["X_test"]), labels=features.LABELS, zero_division=0))

## 17. Error analysis
Which findings are mis-classified, and are **High** findings being missed?

In [ ]:
errors = train.error_analysis(fitted[best], d["X_test"], d["y_test"], d["meta_test"])
errors_table = errors.pop("errors_table")
print(f"{errors['n_errors']} errors out of {errors['n']} test findings")
print("error types:", errors["error_pairs"])
print("errors by rule:", errors["errors_by_rule"])
print("missed High findings:", len(errors["missed_high"]))
display(errors_table.head(15))

## 18. Export the best model
Writes `accessflow_priority_model.joblib` (preprocessing + model in one pipeline), `feature_schema.json` and `model_metadata.json`.
Models from non-real data or failed quality gates are written to `artifacts/dev/` and flagged; only gate-passing real-data models go to `artifacts/`.

In [ ]:
result = train.finalize(DATASET_PATH, d, fitted, cv_summary, best, test_metrics, ci, errors, errors_table, ARTIFACTS_ROOT, seed=SEED)
m = result["metadata"]
print("exported to:", result["dir"])
print("model:", m["model_name"], m["model_version"], "| algorithm:", m["algorithm"])
print("production_eligible:", m["production_eligible"], "|", m["eligibility_reason"])
print("evaluation_status:", m["evaluation_status"])
print(json.dumps(result["paths"], indent=2))